### Train adversarial anomaly detection model
In the previous notebook we performed hyperparamer tuning for adversarial anomaly detection model. Now we are ready to train the model based on the best hyper parameters and export to model repository.
![Training Dataset](./images/experiment_td.png)

In [ ]:
# Setup for local execution
import os
import json
import uuid
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import roc_auc_score, classification_report
import matplotlib.pyplot as plt

# Define paths
BASE_PATH = os.path.dirname(os.path.abspath("__file__"))
TRAINING_DATA_PATH = os.path.join(BASE_PATH, "training_data")
RESOURCES_PATH = os.path.join(BASE_PATH, "Resources")
MODELS_PATH = os.path.join(BASE_PATH, "models")
GAN_DATA_PATH = os.path.join(TRAINING_DATA_PATH, "gan")

# Override paths when running via pipeline (artifacts_dir injected by papermill)
try:
    if artifacts_dir:
        TRAINING_DATA_PATH = os.path.join(artifacts_dir, "data")
        MODELS_PATH = os.path.join(artifacts_dir, "models")
        GAN_DATA_PATH = os.path.join(artifacts_dir, "data", "gan")
except NameError:
    pass

print(f"TensorFlow version: {tf.__version__}")

## Connect to hsfs and retrieve datasets for training and evaluation 

In [2]:
# Load hyperparameters
emb_hp_path = os.path.join(RESOURCES_PATH, "embeddings_best_hp.json")
with open(emb_hp_path, 'r') as f:
    emb_best_hp = json.load(f)

gan_hp_path = os.path.join(RESOURCES_PATH, "gan_best_hp.json")
with open(gan_hp_path, 'r') as f:
    gan_best_hp = json.load(f)

input_dim = emb_best_hp['emb_size']
print(f"Embedding hyperparameters: {emb_best_hp}")
print(f"GAN hyperparameters: {gan_best_hp}")

Embedding hyperparameters: {'walk_number': 2, 'walk_length': 2, 'emb_size': 32}
GAN hyperparameters: {'latent_dim': 8, 'n_layers': 2, 'activation': 'relu', 'dropout_rate': 0.0, 'learning_rate': 0.0001}


### Define hopsworks experiments wrapper function and put all the training logic there. 

In [3]:
# Load training data
X_train = np.load(os.path.join(GAN_DATA_PATH, "X_train.npy"))
y_train = np.load(os.path.join(GAN_DATA_PATH, "y_train.npy"))
X_eval = np.load(os.path.join(GAN_DATA_PATH, "X_eval.npy"))
y_eval = np.load(os.path.join(GAN_DATA_PATH, "y_eval.npy"))

print(f"Training data: {X_train.shape}")
print(f"Evaluation data: {X_eval.shape}")
print(f"Evaluation labels - SAR: {y_eval.sum()}, Non-SAR: {(y_eval==0).sum()}")

Training data: (5224, 32)
Evaluation data: (2123, 32)
Evaluation labels - SAR: 816, Non-SAR: 1307


## Use above experiments wrapper function to conduct hops training experiments.

In [4]:
# Build autoencoder with best hyperparameters
def build_autoencoder(input_dim, latent_dim, n_layers, activation, dropout_rate, learning_rate):
    """Build an autoencoder for anomaly detection."""
    
    # Encoder
    encoder_input = layers.Input(shape=(input_dim,))
    x = encoder_input
    
    units = input_dim
    for i in range(n_layers):
        units = max(units // 2, latent_dim)
        x = layers.Dense(units, activation=activation)(x)
        if dropout_rate > 0:
            x = layers.Dropout(dropout_rate)(x)
    
    latent = layers.Dense(latent_dim, activation=activation, name='latent')(x)
    
    # Decoder
    x = latent
    units = latent_dim
    for i in range(n_layers):
        units = min(units * 2, input_dim)
        x = layers.Dense(units, activation=activation)(x)
        if dropout_rate > 0:
            x = layers.Dropout(dropout_rate)(x)
    
    decoder_output = layers.Dense(input_dim, activation='linear')(x)
    
    # Full autoencoder
    autoencoder = keras.Model(encoder_input, decoder_output, name='autoencoder')
    autoencoder.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mse'
    )
    
    return autoencoder

# Build model
model = build_autoencoder(
    input_dim=input_dim,
    latent_dim=gan_best_hp['latent_dim'],
    n_layers=gan_best_hp['n_layers'],
    activation=gan_best_hp['activation'],
    dropout_rate=gan_best_hp['dropout_rate'],
    learning_rate=gan_best_hp['learning_rate']
)

model.summary()

I0000 00:00:1770024858.617515   40926 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9511 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4080 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Model: "autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ latent (Dense)                  │ (None, 8)              │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         1,056 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,480 (9.69 KB)

 Trainable params: 2,480 (9.69 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# Train the model
EPOCHS = 50
BATCH_SIZE = 32

print("Training anomaly detection model...")
history = model.fit(
    X_train, X_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    verbose=1
)

print("\nTraining complete!")

Training anomaly detection model...
Epoch 1/50


2026-02-02 14:34:46.959964: I external/local_xla/xla/service/service.cc:163] XLA service 0x74bba800b650 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-02-02 14:34:46.959988: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4080 Laptop GPU, Compute Capability 8.9
2026-02-02 14:34:46.977513: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-02-02 14:34:47.109480: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91801


105/147 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 3.2848e-04

I0000 00:00:1770024888.244415   41182 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


147/147 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - loss: 3.2647e-04 - val_loss: 3.2352e-04
Epoch 2/50
147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 3.2414e-04 - val_loss: 3.2081e-04
Epoch 3/50
147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 3.2140e-04 - val_loss: 3.1798e-04
Epoch 4/50
147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 3.1815e-04 - val_loss: 3.1426e-04
Epoch 5/50
147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 3.1463e-04 - val_loss: 3.1094e-04
Epoch 6/50
147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 3.1099e-04 - val_loss: 3.0720e-04
Epoch 7/50
147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 3.0764e-04 - val_loss: 3.0445e-04
Epoch 8/50
147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 3.0460e-04 - val_loss: 3.0123e-04
Epoch 9/50
147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 3.0140e-04 - val_loss: 2.9806e-04
Epoch 10/50
147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.9771e-04 - val_loss: 2.9423e-04
Epoch 11/50
147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.9366e-

In [6]:
# Evaluate the model
def compute_anomaly_score(model, X):
    """Compute reconstruction error as anomaly score."""
    X_pred = model.predict(X, verbose=0)
    mse = np.mean(np.square(X - X_pred), axis=1)
    return mse

# Compute anomaly scores
anomaly_scores = compute_anomaly_score(model, X_eval)

# Calculate AUC
auc = roc_auc_score(y_eval, anomaly_scores)
print(f"Anomaly Detection AUC: {auc:.4f}")

# Find optimal threshold
from sklearn.metrics import precision_recall_curve
precision, recall, thresholds = precision_recall_curve(y_eval, anomaly_scores)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]
print(f"Optimal threshold: {optimal_threshold:.6f}")

Anomaly Detection AUC: 0.5113
Optimal threshold: 0.000152


In [7]:
# Classification report
y_pred = (anomaly_scores > optimal_threshold).astype(int)
print("\nClassification Report:")
print(classification_report(y_eval, y_pred, target_names=['Non-SAR', 'SAR']))


Classification Report:
              precision    recall  f1-score   support

     Non-SAR       0.72      0.01      0.02      1307
         SAR       0.39      0.99      0.56       816

    accuracy                           0.39      2123
   macro avg       0.55      0.50      0.29      2123
weighted avg       0.59      0.39      0.23      2123



In [8]:
# Save the model locally (replaces Hopsworks model registry)
model_id = str(uuid.uuid4())[:8]
model_dir = os.path.join(MODELS_PATH, f"gan_anomaly_{model_id}")
os.makedirs(model_dir, exist_ok=True)

# Save Keras model
model_path = os.path.join(model_dir, "anomaly_detector.keras")
model.save(model_path)
print(f"Saved model to: {model_path}")

# Save metadata
metadata = {
    'hyperparameters': gan_best_hp,
    'embedding_dim': input_dim,
    'metrics': {
        'auc': float(auc),
        'optimal_threshold': float(optimal_threshold),
        'final_loss': float(history.history['loss'][-1])
    }
}
metadata_path = os.path.join(model_dir, "metadata.json")
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"Saved metadata to: {metadata_path}")

# Save threshold for inference
threshold_path = os.path.join(model_dir, "threshold.npy")
np.save(threshold_path, optimal_threshold)

print(f"\n{'='*50}")
print(f"Model saved to: {model_dir}")
print(f"AUC: {auc:.4f}")
print(f"{'='*50}")

Saved model to: /home/adnoman/projects/aml_gan/AMLend2end/models/gan_anomaly_5f5ae592/anomaly_detector.keras
Saved metadata to: /home/adnoman/projects/aml_gan/AMLend2end/models/gan_anomaly_5f5ae592/metadata.json

Model saved to: /home/adnoman/projects/aml_gan/AMLend2end/models/gan_anomaly_5f5ae592
AUC: 0.5113


### Managing experiments
Experiments service provides a unified view of all the experiments run using the `experiment` module.
<br>
As demonstrated in the gif it provides general information about the experiment and the resulting metric. Experiments can be visualized meanwhile or after training in a TensorBoard.
<br>
<br>
![Image7-Monitor.png](./images/experiments.png)